# FoldTrust Benchmark Walkthrough

This notebook demonstrates FoldTrust's benchmark results by loading saved outputs and showing key findings.

**Runtime:** < 30 seconds (no heavy computation; loads pre-computed results)

In [1]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Set paths relative to notebook location
repo_root = Path.cwd().parent.parent
outputs_dir = repo_root / "benchmarks" / "outputs"

## Layer 2: Structure Accuracy

Load summary statistics for MFE/MEA/centroid predictions vs. reference structures.

In [2]:
layer2_summary = pd.read_csv(outputs_dir / "layer2" / "layer2_summary_overall.csv")
layer2_summary

,method,metric,mean,ci_lower,ci_upper
0,mfe,sensitivity,0.638572,0.616291,0.662414
1,mfe,ppv,0.498691,0.477249,0.519685
2,mfe,f1,0.548223,0.526388,0.569166
3,mfe,mcc,0.557308,0.535761,0.577863
4,mea,sensitivity,0.640510,0.618573,0.661756
5,mea,ppv,0.521385,0.500189,0.542901
6,mea,f1,0.563152,0.542036,0.584853
7,mea,mcc,0.571082,0.550277,0.592576
8,centroid,sensitivity,0.622365,0.600220,0.643464
9,centroid,ppv,0.551336,0.529773,0.573630


MEA achieves the best F1 (0.563) with balanced sensitivity and PPV.

## Layer 3: Calibration

Load calibration metrics and tier PPV.

In [3]:
with open(outputs_dir / "layer3" / "layer3_summary.json") as f:
    layer3 = json.load(f)

print(f"ECE: {layer3['ece']:.4f}")
print(f"AUROC: {layer3['auroc']:.4f}")
print(f"AUPRC: {layer3['auprc']:.4f}")
print()

tier_summary = pd.DataFrame(layer3["mfe_tier_summary"])
print("MFE Tier PPV:")
tier_summary[["tier", "pooled_ppv", "total_pairs"]]

ECE: 0.0664
AUROC: 0.8889
AUPRC: 0.6153

MFE Tier PPV:


,tier,pooled_ppv,total_pairs
0,FIRM,0.673814,16417
1,SOFT,0.299366,8518
2,FLOPPY,0.145712,5504


FIRM tier achieves 67% PPV, showing reliable prediction of true pairs.

## Layer 4: SHAPE Agreement

Load SHAPE correlation metrics for SARS-CoV-2 FSE.

In [4]:
layer4_metrics = pd.read_csv(outputs_dir / "layer4_shape" / "per_dataset_metrics.csv")
layer4_metrics[["dataset", "spearman", "spearman_pvalue", "auroc"]]

,dataset,spearman,spearman_pvalue,auroc
0,incarnato_invitro,0.294186,7.680757e-03,0.705215
1,incarnato_invivo,0.354578,1.250670e-03,0.785000
2,pyle,0.163437,1.448702e-01,0.643991
3,zhang_invitro,0.496823,2.386092e-06,0.850340
4,zhang_invivo,0.547276,1.244877e-07,0.863636


icSHAPE datasets (Zhang) show strongest correlation (ρ = 0.50-0.55).

## Layer 5: Robustness

Load temperature and parameter sweep results.

In [5]:
temp_retention = pd.read_csv(outputs_dir / "layer5" / "temperature_stem_retention.csv")

# Compute pooled retention per tier/temperature
temp_grouped = temp_retention.groupby(["tier", "temperature"]).apply(
    lambda x: (x["retention"] * x["n_stems"]).sum() / x["n_stems"].sum(), include_groups=False
).reset_index(name="pooled_retention")

temp_pivot = temp_grouped.pivot_table(
    index="tier", columns="temperature", values="pooled_retention"
).reindex(["FIRM", "SOFT", "FLOPPY"])

print("Temperature Sweep (pooled retention):")
temp_pivot

Temperature Sweep (pooled retention):


temperature,25.0C,30.0C,37C_baseline,42.0C
tier,,,,
FIRM,0.949985,0.949985,1.000000,1.000000
SOFT,0.600007,0.600007,0.733340,0.733340
FLOPPY,0.272718,0.363655,0.363618,0.363618


In [6]:
params_retention = pd.read_csv(outputs_dir / "layer5" / "parameters_stem_retention.csv")

# Compute pooled retention per tier/parameter set
params_grouped = params_retention.groupby(["tier", "parameters"]).apply(
    lambda x: (x["retention"] * x["n_stems"]).sum() / x["n_stems"].sum(), include_groups=False
).reset_index(name="pooled_retention")

params_pivot = params_grouped.pivot_table(
    index="tier", columns="parameters", values="pooled_retention"
).reindex(["FIRM", "SOFT", "FLOPPY"])[
    ["Turner2004_baseline", "Andronescu2007", "Langdon2018"]
]

print("\nParameter Sweep (pooled retention):")
params_pivot


Parameter Sweep (pooled retention):


parameters,Turner2004_baseline,Andronescu2007,Langdon2018
tier,,,
FIRM,1.000000,0.799990,0.600005
SOFT,0.733340,0.066667,0.133340
FLOPPY,0.363618,0.454527,0.090909


FIRM tier robust at 25-42°C (95-100% retention). Alternative parameters reduce retention to 60-80%.

## Live Demo: FSE Prediction

Run FoldTrust on the SARS-CoV-2 frameshift element.

In [7]:
import foldtrust as ft
from foldtrust._core import FoldData
import ViennaRNA as RNA

# SARS-CoV-2 FSE sequence
fse_seq = "UUUAAACGGGUUUGCGGUGUAAGUGCAGCCCGUCUUACACCGUGCGGCACAGGCACUAGUACUGAUGUCGUAUACAGGGCU"

# Create FoldData
fd = FoldData(sequence=fse_seq, name="sars2-fse")

# Run folding using ViennaRNA Python API (no binary needed)
fc = RNA.fold_compound(fse_seq)
mfe_struct, mfe_energy = fc.mfe()
fd.structures["mfe"] = mfe_struct
fd.uns["mfe_energy"] = mfe_energy

# Compute ensemble
ft.tl.compute_ensemble(fd)

# Call tiers
ft.tl.call_tiers(fd)

# Compute unpaired probabilities
ft.tl.compute_unpaired_probs(fd)

print(f"Sequence length: {len(fse_seq)} nt")
print(f"MFE: {fd.uns['mfe_energy']:.2f} kcal/mol")
print(f"MFE structure: {fd.structures['mfe']}")
print()

# Show stem tier summary
if "stems" in fd.uns:
    print(f"Number of stems: {len(fd.uns['stems'])}")
    tier_counts = {}
    for stem in fd.uns["stems"]:
        tier = stem["flag"]  # Use flag not tier
        tier_counts[tier] = tier_counts.get(tier, 0) + 1
    print("\nStem tier counts:")
    for tier in ["FIRM", "SOFT", "FLOPPY", "LONELY"]:
        print(f"  {tier}: {tier_counts.get(tier, 0)}")

Sequence length: 81 nt
MFE: -26.00 kcal/mol
MFE structure: .....(((((((.(((.......))))))))))......(((((((((((((.........))).))))))))...))...

Number of stems: 5

Stem tier counts:
  FIRM: 0
  SOFT: 0
  FLOPPY: 0
  LONELY: 0


In [8]:
# Plot unpaired probability
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(fd.obs["unpaired_prob"], linewidth=1.5)
ax.set_xlabel("Position")
ax.set_ylabel("Unpaired Probability")
ax.set_title("SARS-CoV-2 FSE: Unpaired Probability")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

/tmp/ipykernel_5244/3141721476.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

FoldTrust's benchmark tests:
- **Structure accuracy:** MEA F1 = 0.563 on 600 structures
- **Calibration:** FIRM PPV = 0.674; AUROC = 0.889
- **SHAPE agreement:** ρ = 0.29–0.55 for 4 of 5 datasets on FSE (Pyle ρ=0.16 n.s.)
- **Robustness:** FIRM 95% retention at 25-42°C

Higher tiers have higher reference agreement. See `BENCHMARK.md` for full results.